# Silver preprocessing and demand analysis

## Objective
Convert the wide M5 sales files into one typed row per SKU-store-day, attach calendar and weekly price information, validate the resulting panel, and quantify the demand patterns that will determine model difficulty.

The `dev` profile selects 300 complete series per state using a seeded hash. It does not sample individual rows, because partial histories would distort lags, intermittency, lifecycle, and backtesting. No statistical normalization or target scaling happens in Silver: units and prices retain their business scale. Model-specific encoding and scaling are documented in the Gold and training notebooks.


In [ ]:
profile = "dev"
run_id = "notebook-silver"
force = False
execute_stage = False
seed = 42


## Transformation contract

| Step | Input | Transformation | Output decision |
| --- | --- | --- | --- |
| Series sampling | 30,490 evaluation series | Seeded `xxhash64(id, seed)`, ranked independently inside CA, TX, and WI | Preserves complete histories and state representation |
| Wide-to-long | `d_1` through `d_1941` | Spark `unpivot` | One observation per `series_id` and `day_num` |
| Identity | M5 item and store keys | `series_id = item_id + '_' + store_id` | Stable bottom-level forecast key |
| Calendar join | Daily demand and calendar | Inner join on both `d` and parsed `day_num` | Every demand row must have exactly one date |
| Price join | Daily panel and weekly prices | Left join on store, item, and `wm_yr_wk` | Keeps pre-launch and unavailable-price rows |
| Missing price | Null `sell_price` | Retain null and add `price_missing` | No silent imputation in Silver |
| Physical layout | Typed daily panel | Delta partitioning by state and year | Efficient state and time filtering |

The production implementation is `retail_forecasting.data.silver`; the notebook invokes it and then analyses its Delta outputs rather than reproducing transformation logic in cells.


In [ ]:
from IPython.display import display
import pandas as pd
from pyspark.sql import functions as F

from retail_forecasting.config import load_config
from retail_forecasting.data.silver import run_silver
from retail_forecasting.data.spark import get_spark, table_path

config = load_config(profile)
execute_stage_enabled = str(execute_stage).strip().lower() in {"1", "true", "yes"}
stage_result = run_silver(config, run_id) if execute_stage_enabled else {"status": "reusing existing Silver tables"}
stage_result


In [ ]:
spark = get_spark(config, "notebook-silver-analysis")
daily = spark.read.format("delta").load(str(table_path(config, "silver", "sales_daily"))).cache()
print(f"Silver path: {table_path(config, 'silver', 'sales_daily')}")
print(f"Columns: {len(daily.columns)}")


## Contract and quality gates
A valid panel has non-negative demand, a unique `(series_id, day_num)` key, complete calendar coverage, and at least one row and series. Price nulls are measured separately because they are meaningful and permitted.


In [ ]:
from retail_forecasting.data.quality import validate_silver_sales

quality = validate_silver_sales(daily)
display(pd.DataFrame([quality]))
assert quality["valid"], quality


## Demand distribution
Daily retail demand is non-negative, right-skewed, and frequently zero. Means alone are therefore misleading; the profile reports zero frequency and robust quantiles alongside dispersion.


In [ ]:
overall = daily.agg(
    F.count("*").alias("rows"),
    F.countDistinct("series_id").alias("series"),
    F.sum("units").alias("total_units"),
    F.avg("units").alias("mean_daily_units"),
    F.stddev_pop("units").alias("std_daily_units"),
    F.avg((F.col("units") == 0).cast("double")).alias("zero_rate"),
    F.expr("percentile_approx(units, array(0.5, 0.9, 0.99), 10000)").alias("p50_p90_p99"),
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date"),
).toPandas()
display(overall)


## Intermittency by series
Series are grouped by their historical zero-demand rate. This is diagnostic segmentation, not a feature derived from the future. The same concept is later calculated from trailing windows at each forecast origin.


In [ ]:
activity = (
    daily.groupBy("series_id", "state_id", "cat_id", "dept_id")
    .agg(
        F.avg((F.col("units") == 0).cast("double")).alias("zero_rate"),
        F.sum("units").alias("total_units"),
        F.sum((F.col("units") > 0).cast("int")).alias("selling_days"),
        F.avg(F.when(F.col("units") > 0, F.col("units"))).alias("mean_nonzero_units"),
    )
    .withColumn(
        "demand_regime",
        F.when(F.col("zero_rate") <= 0.50, "dense")
        .when(F.col("zero_rate") <= 0.80, "intermittent")
        .when(F.col("zero_rate") <= 0.95, "sparse")
        .otherwise("very_sparse"),
    )
)
regime_summary = (
    activity.groupBy("demand_regime")
    .agg(
        F.count("*").alias("series"),
        F.sum("total_units").alias("units"),
        F.avg("zero_rate").alias("mean_zero_rate"),
        F.expr("percentile_approx(selling_days, 0.5)").alias("median_selling_days"),
    )
    .orderBy("mean_zero_rate")
    .toPandas()
)
regime_summary["series_share"] = regime_summary["series"] / regime_summary["series"].sum()
regime_summary["demand_share"] = regime_summary["units"] / regime_summary["units"].sum()
display(regime_summary)


## Retail mix
State and category summaries reveal whether aggregate conclusions are dominated by a particular business segment. They also verify that the deterministic development sample contains all intended strata.


In [ ]:
retail_mix = (
    daily.groupBy("state_id", "cat_id")
    .agg(
        F.countDistinct("series_id").alias("series"),
        F.sum("units").alias("units"),
        F.avg("units").alias("mean_daily_units"),
        F.avg((F.col("units") == 0).cast("double")).alias("zero_rate"),
        F.expr("percentile_approx(units, 0.9)").alias("p90_units"),
    )
    .orderBy("state_id", F.desc("units"))
    .toPandas()
)
display(retail_mix)


## Price coverage
A missing weekly price is not automatically an error: M5 commonly has no price before an item is sold in a store. Silver preserves the null and records `price_missing`; Gold fills the numeric model input with zero only after retaining that flag.


In [ ]:
price_profile = (
    daily.groupBy("cat_id")
    .agg(
        F.avg(F.col("price_missing").cast("double")).alias("missing_price_rate"),
        F.expr("percentile_approx(sell_price, 0.5)").alias("median_price"),
        F.expr("percentile_approx(sell_price, 0.9)").alias("p90_price"),
        F.min("sell_price").alias("min_price"),
        F.max("sell_price").alias("max_price"),
    )
    .orderBy("cat_id")
    .toPandas()
)
display(price_profile)


## Demand over time
The monthly trajectory is inspected for lifecycle growth, level shifts, and incomplete periods. It is descriptive only and is not used to fit or choose a model.


In [ ]:
monthly = (
    daily.groupBy(F.trunc("date", "month").alias("month"))
    .agg(F.sum("units").alias("units"))
    .orderBy("month")
    .toPandas()
)
ax = monthly.plot(x="month", y="units", figsize=(12, 4), legend=False, title="Monthly units in the selected profile")
ax.set_ylabel("Units")
ax.grid(axis="y", alpha=0.25)


## Interpretation and limitations

- A zero can mean no customer demand, temporary unavailability, or an item not yet ranged. M5 does not contain on-hand stock, so these mechanisms cannot be separated.
- Weekly price is an imperfect promotion proxy; explicit promotion depth and display information are unavailable.
- The development sample is suitable for workflow iteration, not for claiming a full M5 benchmark. Results must be segmented by demand regime because aggregate point metrics can hide very different behavior.
- No future demand, future-derived aggregate, or globally fitted normalization is created in Silver.


In [ ]:
daily.unpersist()
spark.stop()
